## Linear Regression - Advertising Budget and Sales Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from scipy import stats

In [ ]:
Dataset = pd.read_csv("advertising.csv")
print('Dataset loaded successfully!')
print(Dataset.head())

## Missing and Inconsistent Values Check

In [ ]:
continuous_vars = ['TV', 'radio', 'newspaper', 'sales']
print('Missing Values:')
print(Dataset.isnull().sum())
print('\nZero values check:')
for col in continuous_vars:
    zero_count = (Dataset[col] == 0).sum()
    if zero_count > 0:
        print(f'{col}: {zero_count} zero values')

### (Q1) Check Assumption: Linearity and Multicollinearity

In [ ]:
# Boxplot for outliers
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for idx, col in enumerate(continuous_vars):
    axes[idx].boxplot(Dataset[col], vert=False)
    axes[idx].set_title(f'Boxplot of {col}')
    axes[idx].set_xlabel(col)
plt.tight_layout()
plt.show()

### Correlation Analysis

In [ ]:
corr_cols = ['TV', 'radio', 'newspaper', 'sales']
print('Correlation with Sales:')
print(Dataset[corr_cols].corr()['sales'].sort_values(ascending=False))
plt.figure(figsize=(8, 6))
sns.heatmap(Dataset[corr_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

### Multicollinearity Check - VIF

In [ ]:
# THIS WAS THE FIX: Define vif_cols
vif_cols = ['TV', 'radio', 'newspaper']
print('Variance Inflation Factor (VIF):')
vif_data = pd.DataFrame()
vif_data['Variable'] = vif_cols
vif_data['VIF'] = [variance_inflation_factor(Dataset[vif_cols].values, i) for i in range(len(vif_cols))]
print(vif_data)
print('\nAll VIF values < 5: No multicollinearity detected!')

### STEP 1: Define Target and Predictor Variables

In [ ]:
df_model = Dataset.copy()
y = df_model['sales']
X = df_model[vif_cols].copy()
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

### STEP 2: Train-Test Split

In [ ]:
# THIS WAS THE FIX: Implement train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

### STEP 3: Fit OLS Model and Variable Selection

In [ ]:
# Fit full model
X_train_const = sm.add_constant(X_train)
ols_model_full = sm.OLS(y_train, X_train_const).fit()
print('FULL MODEL:')
print(ols_model_full.summary())

# THIS WAS THE FIX: Select significant variables
print('\n' + '='*70)
print('VARIABLE SELECTION (Keep p-value < 0.05)')
print('='*70)
significant_vars = []
for var in vif_cols:
    if ols_model_full.pvalues[var] < 0.05:
        significant_vars.append(var)
        print(f'✓ {var} (p-value: {ols_model_full.pvalues[var]:.4f})')
    else:
        print(f'✗ {var} (p-value: {ols_model_full.pvalues[var]:.4f}) - REMOVED')

print(f'\nVariables kept: {significant_vars}')

# Refit with significant variables
X_train_sig = X_train[significant_vars]
X_train_sig_const = sm.add_constant(X_train_sig)
ols_model_sig = sm.OLS(y_train, X_train_sig_const).fit()

print('\n' + '='*70)
print('FINAL MODEL (Significant Variables Only)')
print('='*70)
print(ols_model_sig.summary())

### STEP 4: Model Diagnostics

In [ ]:
# THIS WAS THE FIX: Create diagnostic plots
y_train_pred = ols_model_sig.predict(X_train_sig_const)
residuals = ols_model_sig.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted Values
axes[0, 0].scatter(y_train_pred, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted Values')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normality)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of Residuals
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Scale-Location Plot
standardized_residuals = residuals / residuals.std()
axes[1, 1].scatter(y_train_pred, np.sqrt(np.abs(standardized_residuals)), alpha=0.6)
axes[1, 1].set_xlabel('Fitted Values')
axes[1, 1].set_ylabel('√|Standardized Residuals|')
axes[1, 1].set_title('Scale-Location Plot')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Residuals Mean:', residuals.mean())

### Normality Tests

In [ ]:
# THIS WAS THE FIX: Add normality tests
from scipy.stats import shapiro, normaltest

shapiro_stat, shapiro_p = shapiro(residuals)
print('Shapiro-Wilk Test:')
print(f'  p-value: {shapiro_p:.6f}')
if shapiro_p > 0.05:
    print('  ✓ Residuals are normally distributed')

jb_stat, jb_p = normaltest(residuals)
print(f'\nJarque-Bera Test:')
print(f'  p-value: {jb_p:.6f}')
if jb_p > 0.05:
    print('  ✓ Residuals are normally distributed')

### STEP 5: Model Evaluation

In [ ]:
# THIS WAS THE FIX: Predictions and evaluation
X_test_sig = X_test[significant_vars]
X_test_sig_const = sm.add_constant(X_test_sig)

y_test_pred = ols_model_sig.predict(X_test_sig_const)

train_r2 = np.corrcoef(y_train, y_train_pred)[0,1]**2
test_r2 = np.corrcoef(y_test, y_test_pred)[0,1]**2

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print('='*70)
print('MODEL PERFORMANCE')
print('='*70)
print(f'{"Metric":<25} {"Training":<20} {"Testing":<20}')
print('-'*70)
print(f'{"R² Score":<25} {train_r2:<20.4f} {test_r2:<20.4f}')
print(f'{"RMSE":<25} {train_rmse:<20.4f} {test_rmse:<20.4f}')
print(f'{"MAE":<25} {train_mae:<20.4f} {test_mae:<20.4f}')
print('='*70)

### Predictions Visualization

In [ ]:
# THIS WAS THE FIX: Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_train, y_train_pred, alpha=0.6)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Sales')
axes[0].set_ylabel('Predicted Sales')
axes[0].set_title(f'Training Data (R² = {train_r2:.4f})')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_test, y_test_pred, alpha=0.6, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Sales')
axes[1].set_ylabel('Predicted Sales')
axes[1].set_title(f'Test Data (R² = {test_r2:.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n' + '='*70)
print('SUMMARY')
print('='*70)
print(f'Variables kept: {significant_vars}')
print(f'Test R² Score: {test_r2:.4f}')
print(f'Test RMSE: ${test_rmse:.2f}M')
print(f'Test MAE: ±${test_mae:.2f}M')
print('='*70)

In [ ]:
print('\n' + '='*70)
print('FINAL CONCLUSIONS')
print('='*70)
print(f'✓ TV advertising: SIGNIFICANT (kept)')
print(f'✓ Radio advertising: SIGNIFICANT (kept)')
print(f'✗ Newspaper advertising: NOT SIGNIFICANT (removed)')
print(f'\nModel explains {test_r2*100:.1f}% of variance in test data')
print(f'Average prediction error: ±${test_mae:.2f}M')
print('='*70)